# Build your own GPT-4 Tokenizer!

### Step 1

Write the `BasicTokenizer` class, with the following three core functions:

- `def train(self, text, vocab_size, verbose=False)`
- `def encode(self, text)`
- `def decode(self, ids)`

Train your tokenizer on whatever text you like and visualize the merged tokens. Do they look reasonable? One default test you may wish to use is the text file `tests/taylorswift.txt`.

In [20]:
class BasicTokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {idx: bytes([idx]) for idx in range(256)}

    def get_state(self, tokens):
        counts = {}
        chunks = tokens if (tokens and isinstance(tokens[0], list)) else [tokens]
        for chunk in chunks:
            for t1,t2 in zip(chunk[:], chunk[1:]):
                counts[t1,t2] = counts.get((t1,t2),0) + 1
        return counts

    def merge(self, tokens, pair, val):
        new_tokens = []
        i = 0
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                new_tokens.append(val)
                i+=2
            else:
                new_tokens.append(tokens[i])
                i+=1
        return new_tokens

    def train(self, text, vocab_size, is_regex =False):
        if is_regex:
            tokens = [list(chunk.encode("utf-8")) for chunk in text]
        else:
            tokens = list(text.encode("utf-8"))

        nun_merge = vocab_size - 256
        for i in range(nun_merge):
            stats = self.get_state(tokens)
            top_pair = max(stats, key=stats.get)
            if is_regex:
                tokens = [self.merge(chunk, top_pair, 256+i) for chunk in tokens]
            else:
                tokens = self.merge(tokens, top_pair, 256+i)

            self.merges[top_pair] = 256+i
            self.vocab[256+i] = self.vocab[top_pair[0]] + self.vocab[top_pair[1]]

    def encode(self, text):
        tokens = list(text.encode("utf-8"))
        for (t1,t2), val in self.merges.items():
            tokens = self.merge(tokens, (t1,t2), val)
        return tokens

    def decode(self, ids):
        text = b"".join(self.vocab[idx] for idx in ids)
        text = text.decode("utf-8", errors="replace")
        return text

In [21]:
with open("/home/belal/projects/Neural Networks/4.minBPE/train_text.txt", "r", encoding="utf-8") as f:
    train_text = f.read()

tokenizer = BasicTokenizer()
tokenizer.train(train_text, 512)

with open("/home/belal/projects/Neural Networks/4.minBPE/test_text.txt", "r", encoding="utf-8") as f:
    test_text = f.read()

test_text == tokenizer.decode(tokenizer.encode(test_text))

True

### Step 2

Convert you `BasicTokenizer` into a `RegexTokenizer`, which takes a regex pattern and splits the text exactly as GPT-4 would. Process the parts separately as before, then concatenate the results. Retrain your tokenizer and compare the results before and after. You should see that you will now have no tokens that go across categories (numbers, letters, punctuation, more than one whitespace). Use the GPT-4 pattern:

```
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
```

In [22]:
import regex as re

GPT4_SPLIT_PATTERN = re.compile(
    r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
)

class RegexTokenizer:
    def __init__(self):
        self.pattern = GPT4_SPLIT_PATTERN
        self.tokenizer = BasicTokenizer()

    def train(self, text, vocab_size):
        text_chunks = re.findall(self.pattern, text)
        self.tokenizer.train(text_chunks, vocab_size=vocab_size, is_regex=True)

    def encode(self, text):
        text_chunks = re.findall(self.pattern, text)
        all_tokens = []
        for chunk in text_chunks:
            all_tokens.append(self.tokenizer.encode(chunk))
        return all_tokens

    def decode(self, ids):
        return self.tokenizer.decode(ids)

In [25]:
basic_tok = BasicTokenizer()
basic_tok.train(test_text, vocab_size=270)

regex_tok = RegexTokenizer()
regex_tok.train(test_text, vocab_size=270)

print("Basic Tokenizer Merges:")
print([basic_tok.vocab[idx] for idx in range(256, 270) if idx in basic_tok.vocab])

print("\nRegex Tokenizer Merges:")
print([regex_tok.tokenizer.vocab[idx] for idx in range(256, 270) if idx in regex_tok.tokenizer.vocab])

Basic Tokenizer Merges:
[b'e ', b', ', b'd ', b'. ', b'r ', b'20', b's ', b'in', b'on', b'ri', b't ', b'th', b'ed ', b', 20']

Regex Tokenizer Merges:
[b'er', b'20', b'or', b'in', b'ed', b' t', b'on', b'he', b' S', b'ar', b'an', b' A', b' the', b'al']


If you've made it this far, you're now a pro at LLM Tokenization! Sadly, you're not exactly done yet because a lot of LLMs outside of OpenAI (e.g. Llama, Mistral) use [sentencepiece](https://github.com/google/sentencepiece) instead. Primary difference being that sentencepiece runs BPE directly on Unicode code points instead of on UTF-8 encoded bytes. Feel free to explore sentencepiece on your own (good luck, it's not too pretty), and stretch goal if you really experience and suffer from the burden of time, re-write your BPE to be on Unicode code points and match the Llama 2 tokenizer.